<a href="https://colab.research.google.com/github/manaskanugo97-max/Healthcare-Lead-Gen-System/blob/main/merge_final_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/healthcare_lead_gen_project

/content/drive/MyDrive/healthcare_lead_gen_project


In [ ]:
import os

print("Project files:")
print(os.listdir())

print("\nData files:")
print(os.listdir("data"))

Project files:
['data', '__pycache__', 'osm_scraper.py', 'online_presence_checker.py', 'website_analyzer.py', 'scorer.py', 'config.py', 'contact_extractor.py']

Data files:
['healthcare_osm_raw.csv', 'healthcare_online_presence.csv', 'healthcare_website_analysis.csv', 'healthcare_scored_leads.csv', 'healthcare_contact_enriched.csv']


In [ ]:
%%writefile config.py

import os

PROJECT_DIR = "/content/drive/MyDrive/healthcare_lead_gen_project"
DATA_DIR = os.path.join(PROJECT_DIR, "data")

RAW_OSM_CSV = os.path.join(DATA_DIR, "healthcare_osm_raw.csv")
ONLINE_PRESENCE_CSV = os.path.join(DATA_DIR, "healthcare_online_presence.csv")
WEBSITE_ANALYSIS_CSV = os.path.join(DATA_DIR, "healthcare_website_analysis.csv")
SCORED_LEADS_CSV = os.path.join(DATA_DIR, "healthcare_scored_leads.csv")
CONTACT_ENRICHED_CSV = os.path.join(DATA_DIR, "healthcare_contact_enriched.csv")

# Step 6 final output
FINAL_CSV = os.path.join(DATA_DIR, "healthcare_leads_final.csv")

Overwriting config.py


In [ ]:
from config import CONTACT_ENRICHED_CSV, FINAL_CSV
import os

print("Step 6 input:", CONTACT_ENRICHED_CSV)
print("Input exists:", os.path.exists(CONTACT_ENRICHED_CSV))

print("\nFinal output will be:", FINAL_CSV)

Step 6 input: /content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_contact_enriched.csv
Input exists: True

Final output will be: /content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_leads_final.csv


In [ ]:
%%writefile merge_final.py

import os
import pandas as pd
from datetime import datetime

from config import CONTACT_ENRICHED_CSV, FINAL_CSV


def clean_value(value):
    """
    Converts NaN or empty values into clean text.
    """
    if pd.isna(value):
        return ""
    return str(value).strip()


def get_column_value(row, column_name, default=""):
    """
    Safely gets column value from row.
    """
    if column_name in row:
        return clean_value(row[column_name])
    return default


def choose_final_website(row):
    """
    Prefer official website found from Step 2.
    If missing, use original OSM website URL.
    """
    official_website = get_column_value(row, "Official Website Found")
    original_website = get_column_value(row, "Website URL")

    if official_website:
        return official_website

    return original_website


def choose_final_phone(row):
    """
    Prefer final extracted phone.
    If missing, use original phone number.
    """
    final_phone = get_column_value(row, "Final Phone Number")
    original_phone = get_column_value(row, "Phone Number")

    if final_phone:
        return final_phone

    return original_phone


def choose_final_email(row):
    """
    Prefer final extracted email.
    If missing, use original email address.
    """
    final_email = get_column_value(row, "Final Email Address")
    original_email = get_column_value(row, "Email Address")

    if final_email:
        return final_email

    return original_email


def create_final_dataset(input_file=CONTACT_ENRICHED_CSV, output_file=FINAL_CSV):
    """
    Creates final clean healthcare lead dataset.
    """

    print("Step 6 started")
    print("Input file:", input_file)
    print("Output file:", output_file)

    if not os.path.exists(input_file):
        raise FileNotFoundError(
            f"Input file not found: {input_file}. Please complete Step 5 first."
        )

    df = pd.read_csv(input_file)

    final_records = []

    for index, row in df.iterrows():
        website_url = choose_final_website(row)
        phone_number = choose_final_phone(row)
        email_address = choose_final_email(row)

        website_classification = get_column_value(row, "Website Classification")
        business_potential = get_column_value(row, "Business Potential Category")
        lead_score = get_column_value(row, "Lead Score")

        record = {
            # Business Information
            "Business Name": get_column_value(row, "Business Name"),
            "Industry Category": get_column_value(row, "Industry Category"),
            "Business Description": get_column_value(row, "Business Description"),
            "Location": get_column_value(row, "Location"),
            "Google Maps Profile Link": get_column_value(row, "Google Maps Profile Link"),
            "OpenStreetMap Link": get_column_value(row, "OpenStreetMap Link"),

            # Website Analysis
            "Website URL": website_url,
            "Website Classification": website_classification,
            "Website Status Code": get_column_value(row, "Website Status Code"),
            "HTTPS Used": get_column_value(row, "HTTPS Used"),
            "Contact Page Found": get_column_value(row, "Contact Page Found"),
            "Services Page Found": get_column_value(row, "Services Page Found"),
            "Website Analysis Reason": get_column_value(row, "Website Analysis Reason"),
            "Online Presence Type": get_column_value(row, "Online Presence Type"),
            "Referral Platform Links": get_column_value(row, "Referral Platform Links"),

            # Contact Information
            "Phone Number": phone_number,
            "Email Address": email_address,
            "Owner / Founder Name": get_column_value(row, "Owner / Founder Name"),
            "LinkedIn Profile": get_column_value(row, "LinkedIn Profile"),

            # Data Intelligence
            "Lead Score": lead_score,
            "Business Potential Category": business_potential,
            "Reason for Classification": get_column_value(row, "Reason for Classification"),
            "Detailed Score Reason": get_column_value(row, "Detailed Score Reason"),

            # Tracking
            "Source": get_column_value(row, "Source"),
            "Scrape Status": "Completed",
            "Created Date": datetime.now().strftime("%Y-%m-%d")
        }

        final_records.append(record)

    final_df = pd.DataFrame(final_records)

    # Remove exact duplicate leads
    final_df.drop_duplicates(
        subset=["Business Name", "Location", "Google Maps Profile Link"],
        inplace=True
    )

    # Sort high potential leads first
    if "Lead Score" in final_df.columns:
        final_df["Lead Score"] = pd.to_numeric(final_df["Lead Score"], errors="coerce").fillna(0)
        final_df.sort_values(by="Lead Score", ascending=False, inplace=True)

    final_df.to_csv(output_file, index=False, encoding="utf-8-sig")

    print("Step 6 completed")
    print("Final leads:", len(final_df))
    print("Saved final file:", output_file)

    return final_df


if __name__ == "__main__":
    final_df = create_final_dataset(
        input_file=CONTACT_ENRICHED_CSV,
        output_file=FINAL_CSV
    )

    preview_columns = [
        "Business Name",
        "Industry Category",
        "Location",
        "Website URL",
        "Website Classification",
        "Phone Number",
        "Email Address",
        "Lead Score",
        "Business Potential Category",
        "Reason for Classification"
    ]

    available_columns = [col for col in preview_columns if col in final_df.columns]

    print("\nFinal Preview:")
    print(final_df[available_columns].head())

Writing merge_final.py


In [ ]:
!python merge_final.py

Step 6 started
Input file: /content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_contact_enriched.csv
Output file: /content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_leads_final.csv
Step 6 completed
Final leads: 100
Saved final file: /content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_leads_final.csv

Final Preview:
           Business Name  ...                          Reason for Classification
35      Centre For Sight  ...  Business has an online presence but lacks a st...
13  Dr. Vrinda Vashistha  ...  Business has an online presence but lacks a st...
19    Well Care Pharmacy  ...  Business has an online presence but lacks a st...
16  Rashmi Dental Clinic  ...  Business has an online presence but lacks a st...
29       Dr. A. R. pawar  ...  Business has an online presence but lacks a st...

[5 rows x 10 columns]


In [ ]:
import os
import pandas as pd
from config import FINAL_CSV

print("Final CSV exists:", os.path.exists(FINAL_CSV))
print("Final CSV path:", FINAL_CSV)

final_df = pd.read_csv(FINAL_CSV)

print("Total final leads:", len(final_df))

final_df.head()

Final CSV exists: True
Final CSV path: /content/drive/MyDrive/healthcare_lead_gen_project/data/healthcare_leads_final.csv
Total final leads: 100


,Business Name,Industry Category,Business Description,Location,Google Maps Profile Link,OpenStreetMap Link,Website URL,Website Classification,Website Status Code,HTTPS Used,...,Email Address,Owner / Founder Name,LinkedIn Profile,Lead Score,Business Potential Category,Reason for Classification,Detailed Score Reason,Source,Scrape Status,Created Date
0,Centre For Sight,clinic,Clinic listed on OpenStreetMap,"Block no. 124 Section AB, Scheme No. 54, Vijay...",https://www.google.com/maps/search/?api=1&quer...,https://www.openstreetmap.org/node/6187452264,NaN,Poor Website,NaN,NaN,...,NaN,NaN,NaN,65,High,Business has an online presence but lacks a st...,Website exists but digital quality is weak; Bu...,OpenStreetMap,Completed,2026-06-27
1,Dr. Vrinda Vashistha,clinic,Clinic listed on OpenStreetMap,"2-BB, Slice No.-5, Scheme No.-78, Vijay Nagar,...",https://www.google.com/maps/search/?api=1&quer...,https://www.openstreetmap.org/node/6187436057,NaN,Poor Website,NaN,NaN,...,NaN,NaN,NaN,65,High,Business has an online presence but lacks a st...,Website exists but digital quality is weak; Bu...,OpenStreetMap,Completed,2026-06-27
2,Well Care Pharmacy,pharmacy,Pharmacy listed on OpenStreetMap,"EB 303, Scheme No 94, Bombay Hospital Square, ...",https://www.google.com/maps/search/?api=1&quer...,https://www.openstreetmap.org/node/6187441510,NaN,Poor Website,NaN,NaN,...,NaN,NaN,NaN,65,High,Business has an online presence but lacks a st...,Website exists but digital quality is weak; Bu...,OpenStreetMap,Completed,2026-06-27
3,Rashmi Dental Clinic,dentist,Dentist listed on OpenStreetMap,"GH-28, Scheme No. 54, Vijay Nagar, Indore",https://www.google.com/maps/search/?api=1&quer...,https://www.openstreetmap.org/node/6187438667,NaN,Poor Website,NaN,NaN,...,NaN,NaN,NaN,65,High,Business has an online presence but lacks a st...,Website exists but digital quality is weak; Bu...,OpenStreetMap,Completed,2026-06-27
4,Dr. A. R. pawar,clinic,Clinic listed on OpenStreetMap,"AD 335, scheme 74c,vijay nagar, indore",https://www.google.com/maps/search/?api=1&quer...,https://www.openstreetmap.org/node/6187446196,NaN,Poor Website,NaN,NaN,...,NaN,NaN,NaN,65,High,Business has an online presence but lacks a st...,Website exists but digital quality is weak; Bu...,OpenStreetMap,Completed,2026-06-27


In [ ]:
required_columns = [
    "Business Name",
    "Industry Category",
    "Business Description",
    "Location",
    "Google Maps Profile Link",
    "Website URL",
    "Website Classification",
    "Phone Number",
    "Email Address",
    "Owner / Founder Name",
    "LinkedIn Profile",
    "Business Potential Category",
    "Reason for Classification",
    "Lead Score"
]

missing_columns = []

for col in required_columns:
    if col not in final_df.columns:
        missing_columns.append(col)

if len(missing_columns) == 0:
    print("All required columns are present.")
else:
    print("Missing columns:")
    print(missing_columns)

All required columns are present.
